# NES-VMC: Natural Excited States Variational Monte Carlo

## 论文复现: Pfau et al., Science 2024

**论文标题**: Accurate computation of quantum excited states with neural networks

**核心贡献**: 提出了一种无参数的激发态变分蒙特卡洛方法。

本实现采用**分步优化策略**：
1. 先优化基态
2. 固定基态，优化激发态（与基态正交）
3. 计算哈密顿量矩阵对角化得到最终能量

---

In [ ]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '/Users/yangjianfei/mac_vscode/神经网络量子态/激发态能量/Netket_excited_state')
sys.path.insert(0, '.')

import vmc_ex
import nes_vmc

print("="*60)
print("环境信息")
print("="*60)
print(f"NetKet版本: {nk.__version__}")
print(f"JAX版本: {jax.__version__}")

## 1. H2分子系统设置

In [ ]:
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

In [ ]:
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

print("="*60)
print("Hilbert空间信息")
print("="*60)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4")
print(f"电子数: 2 (α=1, β=1)")
print(f"Hilbert空间维度: {hi.n_states}")
print(f"\n所有可能的电子组态:")
print(hi.all_states())

# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

## 2. 神经网络波函数

In [ ]:
class FFN(nnx.Module):
    """前馈神经网络波函数"""
    
    def __init__(self, N: int, alpha: int = 2, *, rngs: nnx.Rngs):
        self.linear1 = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(in_features=alpha * N, out_features=alpha * N, rngs=rngs, param_dtype=complex)
        self.linear_out = nnx.Linear(in_features=alpha * N, out_features=1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array):
        y = self.linear1(x)
        y = nnx.tanh(y)
        y = self.linear2(y)
        y = nnx.tanh(y)
        y = self.linear_out(y)
        return jnp.squeeze(y, axis=-1)

N = 4  # 自旋轨道数
model = FFN(N=N, alpha=2, rngs=nnx.Rngs(42))

params = nnx.state(model, nnx.Param)
n_params = sum(leaf.size for leaf in jtu.tree_leaves(params))

print("="*60)
print("神经网络模型信息")
print("="*60)
print(f"输入维度: {N}")
print(f"隐藏层维度: {N*2}")
print(f"总参数量: {n_params}")

## 3. NES-VMC计算

### 分步优化策略
1. 先优化基态
2. 固定基态，优化激发态（与基态正交）
3. 计算哈密顿量矩阵对角化得到最终能量

In [ ]:
print("="*60)
print("步骤1: 优化基态")
print("="*60)

# 创建基态变分态
model_gs = FFN(N=N, alpha=2, rngs=nnx.Rngs(42))
vs_gs = nk.vqs.MCState(sampler, model_gs, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt_gs = nk.optimizer.Sgd(learning_rate=0.05)
sr_gs = nk.optimizer.SR(diag_shift=0.01)

# 运行基态优化
gs = nk.driver.VMC(ha, opt_gs, variational_state=vs_gs, preconditioner=sr_gs)
print("优化基态 (200迭代)...")
gs.run(n_iter=200, out='h2_ground_state')

E_gs = float(vs_gs.expect(ha).mean.real)
print(f"\n基态能量: {E_gs:.8f} Ha")
print(f"FCI基态: {E_fcis[0]:.8f} Ha")
print(f"误差: {abs(E_gs - E_fcis[0]):.6f} Ha")

In [ ]:
print("\n" + "="*60)
print("步骤2: 优化第一激发态 (与基态正交)")
print("="*60)

# 创建激发态变分态
model_ex1 = FFN(N=N, alpha=2, rngs=nnx.Rngs(142))
vs_ex1 = nk.vqs.MCState(sampler, model_ex1, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt_ex1 = nk.optimizer.Sgd(learning_rate=0.03)
sr_ex1 = nk.optimizer.SR(diag_shift=0.01)

# 使用VMC_ex进行激发态优化（惩罚函数法）
gs_ex1 = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_ex1,
    variational_state=vs_ex1,
    preconditioner=sr_ex1,
    state_list=[vs_gs],  # 与基态正交
    shift_list=[0.3]     # 惩罚参数
)

print("优化第一激发态 (200迭代)...")
gs_ex1.run(n_iter=200, out='h2_excited_state')

E_ex1 = float(vs_ex1.expect(ha).mean.real)
print(f"\n第一激发态能量: {E_ex1:.8f} Ha")
print(f"FCI第一激发态: {E_fcis[1]:.8f} Ha")
print(f"误差: {abs(E_ex1 - E_fcis[1]):.6f} Ha")

In [ ]:
print("\n" + "="*60)
print("步骤3: 计算哈密顿量矩阵并对角化")
print("="*60)

# 计算矩阵
vstate_list = [vs_gs, vs_ex1]
H_matrix, S_matrix = nes_vmc.compute_matrices(vstate_list, ha)

print(f"\n哈密顿量矩阵 H:")
print(H_matrix)

print(f"\n重叠矩阵 S:")
print(S_matrix)

# 对角化
energies, coefficients = nes_vmc.diagonalize_generalized_eigenvalue_problem(H_matrix, S_matrix)

print(f"\n对角化后的能量:")
for i, e in enumerate(energies):
    print(f"  E{i} = {e.real:.8f} Ha (FCI: {E_fcis[i]:.8f} Ha)")

# 检查重叠
overlap = abs(S_matrix[0, 1])
print(f"\n态间重叠 |⟨ψ_0|ψ_1⟩| = {overlap:.6f}")

## 4. 结果分析

In [ ]:
print("="*70)
print("最终结果总结")
print("="*70)

print(f"\n{'态':<15} {'VMC (Ha)':<18} {'FCI (Ha)':<18} {'误差 (Ha)':<15} {'激发能 (eV)'}")
print("-"*85)

for i in range(2):
    if i == 0:
        e_vmc = E_gs
        state_name = "基态"
        exc_eV = 0.0
    else:
        e_vmc = E_ex1
        state_name = "第1激发态"
        exc_eV = (E_ex1 - E_gs) * 27.2114
    
    e_fci = E_fcis[i]
    error = abs(e_vmc - e_fci)
    print(f"{state_name:<15} {e_vmc:<18.8f} {e_fci:<18.8f} {error:<15.8f} {exc_eV:.4f}")

print(f"\n态间重叠: {overlap:.6f}")
if overlap < 0.3:
    print("态区分良好！")
else:
    print("态重叠较大")

In [ ]:
# 绘制能级图
fig, ax = plt.subplots(figsize=(10, 6))

x_vmc = np.arange(2) - 0.15
x_fci = np.arange(2) + 0.15

vmc_energies = [E_gs, E_ex1]
fci_energies = [E_fcis[0], E_fcis[1]]

# 绘制能级
for i in range(2):
    ax.hlines(vmc_energies[i], x_vmc[i]-0.1, x_vmc[i]+0.1, 
              colors='steelblue', linewidth=4, label='VMC' if i==0 else '')
    ax.hlines(fci_energies[i], x_fci[i]-0.1, x_fci[i]+0.1, 
              colors='coral', linewidth=4, linestyle='--', label='FCI' if i==0 else '')
    
    ax.text(x_vmc[i], vmc_energies[i]+0.02, f'{vmc_energies[i]:.4f}', 
            ha='center', fontsize=9, color='steelblue')
    ax.text(x_fci[i], fci_energies[i]-0.04, f'{fci_energies[i]:.4f}', 
            ha='center', fontsize=9, color='coral')

ax.set_xticks(np.arange(2))
ax.set_xticklabels(['E0 (基态)', 'E1 (第1激发态)'])
ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('H2 Energy Levels: VMC vs FCI', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('nes_vmc_energy_levels.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 讨论与总结

### NES-VMC方法特点
1. **分步优化**: 先优化基态，再优化激发态
2. **正交化约束**: 使用惩罚函数确保激发态与基态正交
3. **矩阵对角化**: 计算哈密顿量矩阵得到最终能量

### 本实现与论文的区别
- **论文方法**: 将问题转化为扩展系统的基态问题，无需惩罚参数
- **本实现**: 使用惩罚函数法确保正交性，需要调整惩罚参数

### 改进方向
1. 实现论文中的扩展系统方法
2. 使用更复杂的神经网络架构（如FermiNet）
3. 使用对称性投影加速收敛

In [ ]:
print("="*70)
print("计算完成！")
print("="*70)
print(f"\nH2分子 (键长 = {bond_length} 埃)")
print(f"\n基态:")
print(f"  VMC能量: {E_gs:.8f} Ha")
print(f"  FCI能量: {E_fcis[0]:.8f} Ha")
print(f"  误差:    {abs(E_gs - E_fcis[0]):.6f} Ha")
print(f"\n第一激发态:")
print(f"  VMC能量: {E_ex1:.8f} Ha")
print(f"  FCI能量: {E_fcis[1]:.8f} Ha")
print(f"  误差:    {abs(E_ex1 - E_fcis[1]):.6f} Ha")
print(f"  激发能:  {(E_ex1 - E_gs)*27.2114:.4f} eV (VMC)")
print(f"  激发能:  {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV (FCI)")
print(f"\n态间重叠: {overlap:.6f}")